### Importation des bibliothèques

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter

### CHARGEMENT DES DONNÉES

In [10]:
df = pd.read_csv('../tracks_dataset.csv')
print(f"📊 Dataset chargé : {df.shape[0]} lignes × {df.shape[1]} colonnes")

# Création d'une copie pour la normalisation
df_normalized = df.copy()
print("✅ Copie de travail créée")

📊 Dataset chargé : 887 lignes × 19 colonnes
✅ Copie de travail créée


### VÉRIFICATION ET NORMALISATION DES VALEURS NUMÉRIQUES (0-1)

In [12]:
print("\n🔢 1. NORMALISATION DES CARACTÉRISTIQUES AUDIO")
print("-" * 50)

# Définition des caractéristiques qui doivent être entre 0 et 1
features_0_1 = ['energie', 'valence', 'danceability', 'acousticness', 
                'instrumentalness', 'speechiness', 'liveness']

print("📋 Caractéristiques à vérifier (plage 0-1) :")
for feature in features_0_1:
    print(f"   • {feature}")

print("\n🔍 ÉTAT AVANT NORMALISATION :")
print("-" * 30)

# Analyse des valeurs avant normalisation
issues_found = {}
for feature in features_0_1:
    if feature in df_normalized.columns:
        min_val = df_normalized[feature].min()
        max_val = df_normalized[feature].max()
        out_of_range = ((df_normalized[feature] < 0) | (df_normalized[feature] > 1)).sum()
        
        print(f"{feature:15} | Min: {min_val:6.3f} | Max: {max_val:6.3f} | Hors plage: {out_of_range}")
        
        if min_val < 0 or max_val > 1 or out_of_range > 0:
            issues_found[feature] = {
                'min': min_val,
                'max': max_val,
                'out_of_range': out_of_range
            }

print(f"\n⚠️  Caractéristiques nécessitant une normalisation : {len(issues_found)}")

# Normalisation des caractéristiques hors plage
if issues_found:
    print("\n🔧 NORMALISATION EN COURS :")
    print("-" * 25)
    
    for feature, stats in issues_found.items():
        print(f"\n📊 Normalisation de '{feature}' :")
        
        # Méthode Min-Max scaling pour ramener entre 0 et 1
        old_min = df_normalized[feature].min()
        old_max = df_normalized[feature].max()
        
        if old_max != old_min:  # Éviter la division par zéro
            df_normalized[feature] = (df_normalized[feature] - old_min) / (old_max - old_min)
            print(f"   ✅ [{old_min:.3f}, {old_max:.3f}] → [0.000, 1.000]")
        else:
            df_normalized[feature] = 0  # Si toutes les valeurs sont identiques
            print(f"   ⚠️  Toutes les valeurs identiques ({old_min}) → 0.000")

# Vérification après normalisation
print("\n✅ ÉTAT APRÈS NORMALISATION :")
print("-" * 30)

for feature in features_0_1:
    if feature in df_normalized.columns:
        min_val = df_normalized[feature].min()
        max_val = df_normalized[feature].max()
        out_of_range = ((df_normalized[feature] < 0) | (df_normalized[feature] > 1)).sum()
        print(f"{feature:15} | Min: {min_val:6.3f} | Max: {max_val:6.3f} | Hors plage: {out_of_range}")


🔢 1. NORMALISATION DES CARACTÉRISTIQUES AUDIO
--------------------------------------------------
📋 Caractéristiques à vérifier (plage 0-1) :
   • energie
   • valence
   • danceability
   • acousticness
   • instrumentalness
   • speechiness
   • liveness

🔍 ÉTAT AVANT NORMALISATION :
------------------------------
energie         | Min:  0.217 | Max:  0.950 | Hors plage: 0
valence         | Min:  0.300 | Max:  0.950 | Hors plage: 0
danceability    | Min:  0.236 | Max:  0.950 | Hors plage: 0
acousticness    | Min:  0.230 | Max:  0.900 | Hors plage: 0
instrumentalness | Min:  0.001 | Max:  0.886 | Hors plage: 0
speechiness     | Min:  0.020 | Max:  0.396 | Hors plage: 0
liveness        | Min:  0.050 | Max:  0.350 | Hors plage: 0

⚠️  Caractéristiques nécessitant une normalisation : 0

✅ ÉTAT APRÈS NORMALISATION :
------------------------------
energie         | Min:  0.217 | Max:  0.950 | Hors plage: 0
valence         | Min:  0.300 | Max:  0.950 | Hors plage: 0
danceability    | Min:  

### STANDARDISATION DES NOMS DE CONTEXTES

In [14]:
print("\n🏷️  2. STANDARDISATION DES CONTEXTES")
print("-" * 40)

# Identification des colonnes de contexte
context_columns = [col for col in df_normalized.columns if 'contexte_' in col]
print(f"📋 Colonnes de contexte identifiées : {len(context_columns)}")
for col in context_columns:
    print(f"   • {col}")

def standardize_context_values(text):
    """Standardise les valeurs de contexte"""
    if pd.isna(text):
        return text
    
    # Conversion en string et en minuscules
    text = str(text).lower().strip()
    
    # Suppression des caractères spéciaux et espaces multiples
    text = re.sub(r'[^\w\s-]', '', text)  # Garde lettres, chiffres, espaces et tirets
    text = re.sub(r'\s+', '_', text)      # Remplace espaces multiples par underscore
    text = re.sub(r'-+', '_', text)       # Remplace tirets multiples par underscore
    text = re.sub(r'_+', '_', text)       # Supprime underscores multiples
    text = text.strip('_')                # Supprime underscores en début/fin
    
    return text

print("\n🔧 STANDARDISATION EN COURS :")
print("-" * 30)

# Application de la standardisation à chaque colonne de contexte
standardization_changes = {}

for context_col in context_columns:
    print(f"\n📊 Traitement de '{context_col}' :")
    
    # Valeurs avant standardisation
    unique_before = df_normalized[context_col].unique()
    print(f"   📈 Valeurs avant : {len(unique_before)} uniques")
    
    # Application de la standardisation
    df_normalized[context_col] = df_normalized[context_col].apply(standardize_context_values)
    
    # Valeurs après standardisation
    unique_after = df_normalized[context_col].unique()
    print(f"   📉 Valeurs après : {len(unique_after)} uniques")
    
    # Détection des changements
    changes = []
    for old_val in unique_before:
        if pd.notna(old_val):
            new_val = standardize_context_values(old_val)
            if old_val != new_val:
                changes.append((old_val, new_val))
    
    if changes:
        standardization_changes[context_col] = changes
        print(f"   🔄 Changements effectués : {len(changes)}")
        # Affichage de quelques exemples
        for i, (old, new) in enumerate(changes[:5]):
            print(f"      • '{old}' → '{new}'")
        if len(changes) > 5:
            print(f"      • ... et {len(changes) - 5} autres changements")
    else:
        print("   ✅ Aucun changement nécessaire")


🏷️  2. STANDARDISATION DES CONTEXTES
----------------------------------------
📋 Colonnes de contexte identifiées : 6
   • contexte_temporel
   • contexte_meteo
   • contexte_activite
   • contexte_emotionnel
   • contexte_social
   • contexte_lieu

🔧 STANDARDISATION EN COURS :
------------------------------

📊 Traitement de 'contexte_temporel' :
   📈 Valeurs avant : 6 uniques
   📉 Valeurs après : 6 uniques
   🔄 Changements effectués : 1
      • 'apres-midi' → 'apres_midi'

📊 Traitement de 'contexte_meteo' :
   📈 Valeurs avant : 6 uniques
   📉 Valeurs après : 6 uniques
   ✅ Aucun changement nécessaire

📊 Traitement de 'contexte_activite' :
   📈 Valeurs avant : 8 uniques
   📉 Valeurs après : 8 uniques
   ✅ Aucun changement nécessaire

📊 Traitement de 'contexte_emotionnel' :
   📈 Valeurs avant : 7 uniques
   📉 Valeurs après : 7 uniques
   ✅ Aucun changement nécessaire

📊 Traitement de 'contexte_social' :
   📈 Valeurs avant : 5 uniques
   📉 Valeurs après : 5 uniques
   ✅ Aucun changement 

### CORRECTION DES INCOHÉRENCES DANS LES GENRES MUSICAUX

In [16]:
print("\n🎼 3. CORRECTION DES GENRES MUSICAUX")
print("-" * 40)

if 'genre' in df_normalized.columns:
    print("🔍 ANALYSE DES GENRES AVANT CORRECTION :")
    
    # Analyse des genres actuels
    genre_counts_before = df_normalized['genre'].value_counts()
    print(f"   📊 Genres uniques : {len(genre_counts_before)}")
    print(f"   🎵 Total des tracks : {genre_counts_before.sum()}")
    
    # Affichage des genres problématiques
    rare_genres = genre_counts_before[genre_counts_before <= 5]
    print(f"   ⚠️  Genres rares (≤ 5 tracks) : {len(rare_genres)}")
    
    def standardize_genre(genre):
        """Standardise et corrige les noms de genres"""
        if pd.isna(genre):
            return 'unknown'
        
        genre = str(genre).lower().strip()
        
        # Dictionnaire de mapping pour les corrections communes
        genre_mapping = {
            # Pop variants
            'pop music': 'pop',
            'pop rock': 'pop_rock',
            'pop/rock': 'pop_rock',
            'indie pop': 'indie_pop',
            'synth pop': 'synth_pop',
            'synthpop': 'synth_pop',
            'electropop': 'electro_pop',
            
            # Rock variants
            'rock music': 'rock',
            'hard rock': 'hard_rock',
            'soft rock': 'soft_rock',
            'indie rock': 'indie_rock',
            'alternative rock': 'alternative_rock',
            'classic rock': 'classic_rock',
            
            # Electronic variants
            'electronic music': 'electronic',
            'electronic dance music': 'edm',
            'electro house': 'electro_house',
            'deep house': 'deep_house',
            'tech house': 'tech_house',
            
            # Hip-Hop variants
            'hip hop': 'hip_hop',
            'hip-hop': 'hip_hop',
            'rap music': 'rap',
            'trap music': 'trap',
            
            # Jazz variants
            'jazz music': 'jazz',
            'smooth jazz': 'smooth_jazz',
            'jazz fusion': 'jazz_fusion',
            
            # Classical variants
            'classical music': 'classical',
            'classical crossover': 'classical_crossover',
            
            # R&B variants
            'r&b': 'rnb',
            'r & b': 'rnb',
            'rhythm and blues': 'rnb',
            'contemporary r&b': 'contemporary_rnb',
            
            # Folk variants
            'folk music': 'folk',
            'folk rock': 'folk_rock',
            
            # Country variants
            'country music': 'country',
            'country rock': 'country_rock',
            
            # Ambient variants
            'ambient music': 'ambient',
            'new age': 'new_age',
            
            # Correction des erreurs communes
            '': 'unknown',
            'n/a': 'unknown',
            'na': 'unknown',
            'null': 'unknown',
            'none': 'unknown',
        }
        
        # Application du mapping
        if genre in genre_mapping:
            return genre_mapping[genre]
        
        # Standardisation générale
        genre = re.sub(r'[^\w\s]', '', genre)  # Supprime caractères spéciaux
        genre = re.sub(r'\s+', '_', genre)     # Remplace espaces par underscores
        
        return genre
    
    print("\n🔧 CORRECTION EN COURS :")
    print("-" * 25)
    
    # Application de la standardisation des genres
    df_normalized['genre'] = df_normalized['genre'].apply(standardize_genre)
    
    # Analyse après correction
    genre_counts_after = df_normalized['genre'].value_counts()
    print(f"✅ Genres après correction : {len(genre_counts_after)}")
    
    # Regroupement des genres rares
    rare_threshold = 3  # Seuil pour considérer un genre comme rare
    rare_genres_after = genre_counts_after[genre_counts_after <= rare_threshold]
    
    if len(rare_genres_after) > 0:
        print(f"\n🔄 REGROUPEMENT DES GENRES RARES :")
        print(f"   📊 Genres avec ≤ {rare_threshold} tracks : {len(rare_genres_after)}")
        
        # Regroupement par catégories principales
        genre_categories = {
            'electronic': ['ambient', 'synthwave', 'chillwave', 'downtempo', 'trip_hop'],
            'rock_alternative': ['post_rock', 'math_rock', 'shoegaze', 'garage_rock'],
            'experimental': ['noise', 'avant_garde', 'industrial', 'drone'],
            'world': ['latin', 'reggae', 'african', 'asian', 'celtic'],
            'classical_extended': ['opera', 'chamber', 'orchestral', 'contemporary_classical'],
            'other': []  # Pour tout ce qui ne rentre pas dans les catégories
        }
        
        # Application du regroupement
        def categorize_rare_genre(genre):
            if genre_counts_after[genre] > rare_threshold:
                return genre
            
            # Recherche dans les catégories
            for category, keywords in genre_categories.items():
                if category == 'other':
                    continue
                for keyword in keywords:
                    if keyword in genre:
                        return category
            
            # Si aucune catégorie trouvée, mettre dans "other"
            return 'other'
        
        df_normalized['genre'] = df_normalized['genre'].apply(categorize_rare_genre)
        
        # Statistiques finales
        genre_counts_final = df_normalized['genre'].value_counts()
        print(f"   ✅ Genres finaux : {len(genre_counts_final)}")
        print(f"   📈 Réduction : {len(genre_counts_before)} → {len(genre_counts_final)}")


🎼 3. CORRECTION DES GENRES MUSICAUX
----------------------------------------
🔍 ANALYSE DES GENRES AVANT CORRECTION :
   📊 Genres uniques : 12
   🎵 Total des tracks : 887
   ⚠️  Genres rares (≤ 5 tracks) : 0

🔧 CORRECTION EN COURS :
-------------------------
✅ Genres après correction : 12


### GESTION DES VALEURS ABERRANTES (CARACTÉRISTIQUES AUDIO)

In [18]:
print("\n⚡ 4. GESTION DES VALEURS ABERRANTES")
print("-" * 40)

# Caractéristiques avec plages spécifiques
special_features = {
    'bpm_estime': (40, 200),    # BPM réaliste
    'notes': (0, 10)            # Notes sur 10
}

print("🔍 DÉTECTION DES VALEURS ABERRANTES :")
print("-" * 35)

outliers_detected = {}

for feature, (min_expected, max_expected) in special_features.items():
    if feature in df_normalized.columns:
        # Détection des outliers
        outliers_low = df_normalized[feature] < min_expected
        outliers_high = df_normalized[feature] > max_expected
        total_outliers = outliers_low.sum() + outliers_high.sum()
        
        print(f"\n📊 {feature} (plage attendue: {min_expected}-{max_expected}) :")
        print(f"   📉 Valeurs trop basses (< {min_expected}) : {outliers_low.sum()}")
        print(f"   📈 Valeurs trop hautes (> {max_expected}) : {outliers_high.sum()}")
        print(f"   ⚠️  Total aberrantes : {total_outliers}")
        
        if total_outliers > 0:
            outliers_detected[feature] = {
                'low': outliers_low.sum(),
                'high': outliers_high.sum(),
                'indices_low': df_normalized[outliers_low].index.tolist(),
                'indices_high': df_normalized[outliers_high].index.tolist()
            }

# Traitement des valeurs aberrantes
if outliers_detected:
    print(f"\n🔧 TRAITEMENT DES VALEURS ABERRANTES :")
    print("-" * 35)
    
    for feature, outlier_info in outliers_detected.items():
        min_expected, max_expected = special_features[feature]
        
        print(f"\n📊 Traitement de '{feature}' :")
        
        # Méthode : Clipping (limitation aux valeurs min/max acceptables)
        original_values = df_normalized[feature].copy()
        
        # Clipping des valeurs
        df_normalized[feature] = df_normalized[feature].clip(lower=min_expected, upper=max_expected)
        
        # Statistiques du traitement
        changes = (original_values != df_normalized[feature]).sum()
        print(f"   ✅ Valeurs corrigées : {changes}")
        print(f"   📊 Nouvelle plage : [{df_normalized[feature].min():.1f}, {df_normalized[feature].max():.1f}]")

# Détection d'outliers statistiques pour les autres caractéristiques
print(f"\n📊 DÉTECTION D'OUTLIERS STATISTIQUES (méthode IQR) :")
print("-" * 50)

features_for_outlier_detection = ['energie', 'valence', 'danceability', 'acousticness']

for feature in features_for_outlier_detection:
    if feature in df_normalized.columns:
        Q1 = df_normalized[feature].quantile(0.25)
        Q3 = df_normalized[feature].quantile(0.75)
        IQR = Q3 - Q1
        
        # Calcul des limites (méthode IQR)
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        # Détection des outliers
        outliers = ((df_normalized[feature] < lower_bound) | (df_normalized[feature] > upper_bound))
        outliers_count = outliers.sum()
        
        print(f"{feature:15} | Outliers: {outliers_count:4d} ({(outliers_count/len(df_normalized)*100):5.2f}%)")
        
        # Si trop d'outliers (>5%), on peut choisir de les normaliser
        if outliers_count > len(df_normalized) * 0.05:  # Plus de 5%
            print(f"                | ⚠️  Taux élevé d'outliers détecté")


⚡ 4. GESTION DES VALEURS ABERRANTES
----------------------------------------
🔍 DÉTECTION DES VALEURS ABERRANTES :
-----------------------------------

📊 bpm_estime (plage attendue: 40-200) :
   📉 Valeurs trop basses (< 40) : 0
   📈 Valeurs trop hautes (> 200) : 0
   ⚠️  Total aberrantes : 0

📊 notes (plage attendue: 0-10) :
   📉 Valeurs trop basses (< 0) : 0
   📈 Valeurs trop hautes (> 10) : 0
   ⚠️  Total aberrantes : 0

📊 DÉTECTION D'OUTLIERS STATISTIQUES (méthode IQR) :
--------------------------------------------------
energie         | Outliers:   17 ( 1.92%)
valence         | Outliers:   21 ( 2.37%)
danceability    | Outliers:   25 ( 2.82%)
acousticness    | Outliers:    0 ( 0.00%)


### RÉSUMÉ DE LA NORMALISATION

In [20]:
print("\n📋 RÉSUMÉ DE LA NORMALISATION")
print("=" * 50)

print("✅ TÂCHES ACCOMPLIES :")
print(f"   🔢 Caractéristiques audio normalisées (0-1) : {len([f for f in features_0_1 if f in df_normalized.columns])}")
print(f"   🏷️  Contextes standardisés : {len(context_columns)}")
print(f"   🎼 Genres musicaux corrigés")
print(f"   ⚡ Valeurs aberrantes traitées : {len(outliers_detected)} caractéristiques")

print(f"\n📊 STATISTIQUES FINALES :")
print(f"   📈 Lignes traitées : {len(df_normalized)}")
print(f"   📋 Colonnes : {len(df_normalized.columns)}")
print(f"   🎵 Genres uniques : {df_normalized['genre'].nunique() if 'genre' in df_normalized.columns else 'N/A'}")

# Vérification de l'intégrité des données
print(f"\n🔍 VÉRIFICATION D'INTÉGRITÉ :")
print(f"   💾 Valeurs manquantes : {df_normalized.isnull().sum().sum()}")
print(f"   🔄 Doublons : {df_normalized.duplicated().sum()}")

# Sauvegarde du dataset normalisé
output_filename = '../tracks_dataset_normalized.csv'
df_normalized.to_csv(output_filename, index=False)
print(f"\n💾 SAUVEGARDE :")
print(f"   ✅ Dataset normalisé sauvegardé : '{output_filename}'")
print(f"   📊 Taille du fichier : {len(df_normalized)} lignes")

print(f"\n🎉 NORMALISATION TERMINÉE AVEC SUCCÈS !")
print("   Prêt pour la phase suivante : Analyse exploratoire")


📋 RÉSUMÉ DE LA NORMALISATION
✅ TÂCHES ACCOMPLIES :
   🔢 Caractéristiques audio normalisées (0-1) : 7
   🏷️  Contextes standardisés : 6
   🎼 Genres musicaux corrigés
   ⚡ Valeurs aberrantes traitées : 0 caractéristiques

📊 STATISTIQUES FINALES :
   📈 Lignes traitées : 887
   📋 Colonnes : 19
   🎵 Genres uniques : 12

🔍 VÉRIFICATION D'INTÉGRITÉ :
   💾 Valeurs manquantes : 0
   🔄 Doublons : 0

💾 SAUVEGARDE :
   ✅ Dataset normalisé sauvegardé : '../tracks_dataset_normalized.csv'
   📊 Taille du fichier : 887 lignes

🎉 NORMALISATION TERMINÉE AVEC SUCCÈS !
   Prêt pour la phase suivante : Analyse exploratoire
